# Enveda CASMI26 — R45: unpruned native pool + 2-worker CPU multi-process + A1

R43 restored the real `gen-p2 v8` mainline and scored LB `0.096`, but its CPU run left
ICEBERG at `31/371` covered molecules and GLACIER at `142/371` because all 20,739
candidates were thrown at both networks simultaneously. R44 tried to fix the throughput
problem with a `base_score U sig_ms2` Top-32 pre-filter; it did cover 100% of molecules,
but the pre-filter silently dropped `gen-p2 v8`'s own truths: only **24/40** Rank-1 truths
and **100/209** in-25 truths survived, giving `vis400 MRR@25 = 0.09954`.

**R45 removes the broken pre-filter and runs the native pool in parallel instead.**

| stage | what runs | candidate pool | writes |
|---|---|---|---|
| **1** ~1 s | legal 25-slot floor file | 25 inlined SMILES | `stage1_floor` |
| **2** ~40 s | M-S2..M-S9 (`gen-p2 v8` parser → ±10 ppm pool → 7GR formulas → Gen-A → `base_score`) | all pool / Gen-A candidates | `stage2_base` |
| **3a** ~30 min | CPU **GLACIER** on the native `ice_candidates(base_rank(m), 60)` pool, **2 workers × 2 threads** | **20,739** candidates, 371/371 molecules | `stage3a_gl_native` |
| **3b** ~2 h | CPU **ICEBERG** on the **same** native pool, **2 workers × 2 threads**, independent workdirs + `TMPDIR` | **20,739** candidates, 371/371 molecules | `stage3b_genp2_native` |
| **4** ~3 min | M-S12 `rerank_multi` on the **full** `base_rank(m)` list → M-S13 `A1`: rank 1 frozen, ranks 2..25 re-sorted by `sig_ms2` (9,600 pairs, ~175 s) | 25 | `stage4_A1_final` |

**Why the native pool guarantees zero truth loss.** `gen-p2 v8` M-S10/M-S11 build
`cands_by_mid[mid] = ice_candidates(base_rank(mols[mid]), TOP_N_RESCORE=60)`. R43 used
exactly this and measured `n_cands = 20,739`, `n_mols_covered = 371`, and the resulting
`vis400 = 0.16988 / top1 = 40 / in25 = 209`. Therefore, as long as R45 does **not** add
any extra Top-K truncation, the pool contains the same 40 Rank-1 truths and 209 in-25
truths by set identity — verified inside the kernel by `recall_audit`.

**R44's two root causes, fixed here.**
1. R44 used a single-column `molecule_id` frame for ICEBERG input, which stripped the 12
   spectral metadata columns that `build_ice_input` needs; R45 uses `test_ice = test.copy()`
   exactly like `enveda-gen-p2.ipynb:1411-1420`, so ICEBERG covers 371/371 molecules instead
   of 0/371.
2. The `base_score` / `sig_ms2` Top-32 pre-filter was a weak signal acting as a strong
   selector; R45 deletes it entirely and lets GLACIER / ICEBERG score every native
candidate.

**Engineering.** `N_WORKERS = 2`, each worker `OMP/MKL/torch = 2 threads` (2×2 = 4 vCPU,
well below the 30 GB RAM ceiling). Every worker has its own `workdir` and its own
`TMPDIR/TEMP/TMP` to avoid colliding on `/tmp/ice_ms_pred_cache`. If any shard throws or
scores fewer than ~90% of its expected molecules, a single-process fallback rescues the
missing ones. Zero runtime guard statements of any kind in every generated cell; `submission.csv` is written
atomically (`tempfile` + `os.replace`) five times.

The in-kernel evaluator uses a zlib+base64 diagnostic reference table (`400` ids,
sha256 `ca145a39507748a0`, 40 / 209 sanity-checked at build time); it is read-only and never
contributes a candidate.


In [ ]:
# ===== E0: environment + the ONE atomic-write primitive used by every stage =====
import os
import sys
import glob
import json
import math
import time
import re
import shutil
import tempfile
import traceback
import subprocess
import collections
import base64
import zlib
import hashlib

os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'

import numpy as np
import pandas as pd

T0 = time.time()
WORK = '/kaggle/working'
SUB = os.path.join(WORK, 'submission.csv')
os.makedirs(WORK, exist_ok=True)
STAGES = []
WRITES = []
MENU = []


def log(*a):
    print('[R45 %7.2fs]' % (time.time() - T0), *a, flush=True)


class stage(object):
    """wall-clock record per stage; over-budget is REPORTED, never raised."""
    def __init__(self, name, budget_s=None):
        self.name = name
        self.budget_s = budget_s

    def __enter__(self):
        self.t = time.time()
        log('>> %s' % self.name)
        return self

    def __exit__(self, *exc):
        d = time.time() - self.t
        rec = dict(stage=self.name, seconds=round(d, 1),
                   over_budget=bool(self.budget_s and d > self.budget_s))
        STAGES.append(rec)
        log('<< %s %.1fs%s' % (self.name, d, '  [OVER BUDGET]' if rec['over_budget'] else ''))
        return False


def write_sub(rows, tag, n_ids=None):
    """ATOMIC submission write: tempfile in the same dir + os.replace. Never a partial file."""
    try:
        body = 'molecule_id,smiles\n' + ''.join('%s,%s\n' % (m, ';'.join(s)) for m, s in rows)
        fd, tmp = tempfile.mkstemp(dir=WORK, prefix='.sub_%s_' % tag, suffix='.csv')
        with os.fdopen(fd, 'w', newline='') as fh:
            fh.write(body)
        os.replace(tmp, SUB)
        n = len(rows)
        WRITES.append(dict(tag=tag, seconds=round(time.time() - T0, 2), rows=n,
                           bytes=int(os.path.getsize(SUB))))
        log('WRITE[%s] %s | %d rows | %d bytes%s' % (
            tag, SUB, n, os.path.getsize(SUB),
            '' if n_ids is None or n == n_ids else '  [WARN %d != %d ids]' % (n, n_ids)))
        return True
    except Exception:
        traceback.print_exc()
        return False


try:
    import torch
    torch.set_num_threads(4)
    TORCH_VER = torch.__version__
except Exception:
    TORCH_VER = 'absent'
log('python %s | numpy %s | pandas %s | torch %s | threads 4 (OMP/MKL/torch)' % (
    sys.version.split()[0], np.__version__, pd.__version__, TORCH_VER))


In [ ]:
# ===== S0: gen-p2 v8 config (VERBATIM, CPU budgets overridden for R45) =====
# S0: config
import os, sys, json, math, time, glob
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

CFG = dict(
    COMP_DIR='/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra',
    POOL_GLOB='/kaggle/input/**/pool_meta.parquet',
    EVAL_N=60,
    SEED=42,
    MASS_PPM=10.0,
    MASS_PPM_FALLBACK=20.0,
    FORMULA_TOL_DA=0.02,
    HALOGENS=False,
    FORMULA_CAP=400,
    TOP_N_RESCORE=60,
    ICE_BUDGET_S=4800.0,
    GL_BUDGET_S=4200.0,
    LAM_ICE=0.5,
    LAM_GL=0.5,
    GEN_DERIVE=True,
    GEN_PARENTS=20,
    GEN_CAP_PARENT=60,
    GEN_TOL_DA=0.01,
    TOP_K_SUBMIT=25,
    MASS_BINS=[50, 200, 350, 500, 800, 3000],
)
t0 = time.time()
print('config ok')

# ---- R45: native pool, 2-worker CPU multi-process. ICEBERG gets 5 h for 20,739 candidates;
# GLACIER gets 2 h. Both are well inside the 12 h CPU allowance.
CFG['ICE_BUDGET_S'] = 18000.0
CFG['GL_BUDGET_S'] = 7200.0
CFG['TOP_N_RESCORE'] = 60
N_WORKERS = 2
OMP_PER_WORKER = 2
log('budgets: ICE %ss | GL %ss | TOP_N_RESCORE %d | N_WORKERS %d | threads/worker %d' % (
    CFG['ICE_BUDGET_S'], CFG['GL_BUDGET_S'], CFG['TOP_N_RESCORE'], N_WORKERS, OMP_PER_WORKER))


In [ ]:
# ===== S1: locate the mounted competition data / pool / ICEBERG / GLACIER (VERBATIM) =====
# S1: locate assets
def find_asset(pattern):
    hits = sorted(glob.glob(pattern, recursive=True), key=len)
    return hits[0] if hits else None

TRAIN_PQ = str(Path(CFG['COMP_DIR']) / 'train.parquet')
if not os.path.exists(TRAIN_PQ):
    alt = find_asset('/kaggle/input/**/train.parquet')
    TRAIN_PQ = alt if alt else TRAIN_PQ
POOL_PQ = find_asset(CFG['POOL_GLOB'])
ICE_RUNNER = find_asset('/kaggle/input/**/casmi26-iceberg/**/ice_runner.py') or find_asset('/kaggle/input/**/ice_runner.py')
GL_RUNNER = find_asset('/kaggle/input/**/casmi26-glacier/**/gl_runner.py') or find_asset('/kaggle/input/**/gl_runner.py')
ICE_PKG = os.path.dirname(ICE_RUNNER) if ICE_RUNNER else None
GL_PKG = os.path.dirname(GL_RUNNER) if GL_RUNNER else None
HAS_GPU = bool(glob.glob('/dev/nvidia*')) or os.path.exists('/proc/driver/nvidia')
print('train:', TRAIN_PQ, os.path.exists(TRAIN_PQ))
print('pool:', POOL_PQ)
print('ice pkg:', ICE_PKG)
print('gl pkg:', GL_PKG)
print('gpu:', HAS_GPU)


TEST_PQ = str(Path(CFG['COMP_DIR']) / 'test.parquet')
if not os.path.exists(TEST_PQ):
    _alt = find_asset('/kaggle/input/**/test.parquet')
    TEST_PQ = _alt if _alt else TEST_PQ
print('test:', TEST_PQ, os.path.exists(TEST_PQ))

COMP = os.path.dirname(TEST_PQ) if os.path.exists(TEST_PQ) else None
log('assets: test=%s | pool=%s | ice=%s | gl=%s | gpu=%s' % (
    TEST_PQ, POOL_PQ, ICE_PKG, GL_PKG, HAS_GPU))


In [ ]:
# ===== S2: rdkit (VERBATIM from gen-p2 v8; cp310/cp312/cp313 wheel auto-match) =====
# S2: rdkit (v7 fixed install: keep PEP 427 wheel filename, strip only the trailing .ice)
import glob as _g, os as _os, subprocess as _sp, sys as _sys
try:
    import rdkit
    RDKIT_OK = True
except ImportError:
    _whl = sorted(_g.glob('/kaggle/input/**/rdkit*.whl*', recursive=True))
    RDKIT_OK = False
    if _whl:
        import shutil as _sh
        # keep PEP 427 wheel filename (strip the trailing .ice only) -- pip rejects invalid names
        _dst = '/kaggle/working/' + _os.path.basename(_whl[0]).removesuffix('.ice')
        _sh.copy(_whl[0], _dst)
        _rc = _sp.run([_sys.executable, '-m', 'pip', 'install', '--no-index', '--quiet', _dst]).returncode
        try:
            import rdkit
            RDKIT_OK = True
        except Exception:
            RDKIT_OK = False
    if not RDKIT_OK:
        print('[warn] rdkit unavailable -> Gen-A disabled; eval restricted to exact-SMILES hits')

if RDKIT_OK:
    from rdkit import Chem
    from rdkit.Chem import rdMolDescriptors
    from rdkit.Chem import rdChemReactions
    from rdkit.Chem import inchi as rdi
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')

    def canon_smiles(smi):
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol) if mol is not None else None

    def mol_formula(smi):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        try:
            return rdMolDescriptors.CalcMolFormula(mol)
        except Exception:
            return None

    def formula_ok(smi, formula_set):
        if not formula_set:
            return True
        f = mol_formula(smi)
        return f is not None and f in formula_set

    _IK_CACHE = {}
    def inchikey14(smi):
        # skeleton layer (first 14 chars) of the standard InChIKey
        if smi in _IK_CACHE:
            return _IK_CACHE[smi]
        k = None
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            try:
                ik = rdi.MolToInchiKey(mol)
                k = ik[:14] if ik else None
            except Exception:
                k = None
        _IK_CACHE[smi] = k
        return k
    print('rdkit helpers ok (rdkit', rdkit.__version__ + ')')
else:
    def canon_smiles(smi):
        return smi
    def mol_formula(smi):
        return None
    def formula_ok(smi, formula_set):
        return True
    _IK_CACHE = {}
    def inchikey14(smi):
        return None

try:
    import torch as _t
    _t.set_num_threads(4)
except Exception:
    pass
log('rdkit ok=%s' % (RDKIT_OK,))


In [ ]:
# ===== STAGE 1 (t ~ 1 s): a LEGAL floor submission.csv before any heavy work =====
FLOOR_SMILES = [
    'CCO', 'CC(=O)O', 'CO', 'CN', 'O', 'N', 'COC', 'CC(C)O', 'CC(=O)N', 'c1ccccc1',
    'c1ccccc1O', 'Cc1ccccc1', 'OCC(O)CO', 'CC(=O)OC', 'CCOC(C)=O', 'O=C(O)c1ccccc1',
    'NC(=O)C', 'CCN', 'CCN(CC)CC', 'O=C(N)N', 'CS(=O)(=O)C', 'Cc1ccccc1O', 'OC(=O)C(=O)O',
    'CC(C)(C)O', 'c1ccncc1',
]
FLOOR_SMILES = list(dict.fromkeys([s for s in FLOOR_SMILES if isinstance(s, str) and s]))[:25]

IDS = []
try:
    _t = pd.read_parquet(TEST_PQ)
    IDS = sorted(set(str(v) for v in _t['molecule_id'].tolist()))
    log('test.parquet: %d spectra rows | %d molecule ids' % (len(_t), len(IDS)))
    del _t
except Exception:
    traceback.print_exc()
    log('[warn] test.parquet unreadable -> cannot enumerate ids')

if IDS:
    write_sub([(m, list(FLOOR_SMILES)) for m in IDS], 'stage1_floor', len(IDS))
else:
    write_sub([], 'stage1_floor')
log('STAGE 1 done in %.2fs' % (time.time() - T0))


In [ ]:
# ===== S3: elemental masses + electron (VERBATIM; the ION2MASS hard table is NOT used) =====
# S3: adduct normalization -> neutral monoisotopic mass (ms-pred ion2mass convention; same as gen-p2)
ELECTRON = 0.00054858
EMASS = {'C': 12.0, 'H': 1.00782503223, 'N': 14.00307400443, 'O': 15.99491461957,
         'P': 30.97376199842, 'S': 31.9720711744, 'F': 18.99840316273,
         'Cl': 34.968852682, 'Br': 78.9183376}


log("mass constants ready")


In [ ]:
# ===== S3b: general adduct parser (VERBATIM; gen-p2 v8 mainline mass path) =====
# S3b: general adduct parser -- mainline neutral-mass path, REPLACES the ION2MASS hard table.
# [kM + ligand deltas]z  ->  M = (|z|*mz - D + sign(z)*|z|*ELECTRON) / k
# Supports oligomers (2M, 3M), multi-charge (z=2,3, no digit = 1) and digit-prefixed ligands.
import re
EMASS_V5 = dict(EMASS)
EMASS_V5.setdefault('Na', 22.98976928)
EMASS_V5.setdefault('K', 38.96370668)
EMASS_V5.setdefault('I', 126.9044719)

ADDUCT_RE = re.compile(r'^\[(.+)\](\d*)([+-])$')

def _formula_mass(fstr, emass=None):
    emass = emass or EMASS_V5
    parts = re.findall(r'([A-Z][a-z]?)(\d*)', fstr)
    if ''.join(e + n for e, n in parts) != fstr:
        return None
    m = 0.0
    for e, n in parts:
        if e not in emass:
            return None
        m += emass[e] * (int(n) if n else 1)
    return m

def parse_adduct(adduct):
    """Parse '[2M+Na-2H]-' style strings -> dict(k=mult, z=charge, D=ligand mass delta, known)."""
    if not isinstance(adduct, str):
        return None
    mt = ADDUCT_RE.match(adduct.strip())
    if not mt:
        return None
    inner, zs, sign = mt.groups()
    z = int(zs) if zs else 1
    if sign == '-':
        z = -z
    k, D, known = 1, 0.0, True
    for tok in re.findall(r'[+-]?[^+-]+', inner):
        t = tok
        s = 1.0
        if t[0] == '+':
            t = t[1:]
        elif t[0] == '-':
            s, t = -1.0, t[1:]
        if not t:
            continue
        if t == 'M':
            continue
        if t.endswith('M') and t[:-1].isdigit():
            k *= int(t[:-1])
            continue
        mult = 1
        while t and t[0].isdigit():
            mult = int(t[0]) * mult
            t = t[1:]
        fm = _formula_mass(t)
        if fm is None:
            known = False
            continue
        D += s * mult * fm
    return dict(k=k, z=z, D=D, known=known)

def mz_to_mass(mz, k, z, D):
    az = abs(z)
    sg = 1.0 if z > 0 else -1.0
    return (az * mz - D + sg * az * ELECTRON) / k

def neutral_mass_parser(precursor_mz, adduct, mode):
    """Mainline mass: exact parser first; unparseable (e.g. [Cat]2+) falls back to mode proton delta."""
    p = parse_adduct(adduct)
    if p and p['known'] and np.isfinite(precursor_mz):
        return mz_to_mass(precursor_mz, p['k'], p['z'], p['D'])
    if mode == 'negative':
        return precursor_mz + EMASS['H'] - ELECTRON
    return precursor_mz - EMASS['H'] + ELECTRON

print('general adduct parser ready (mainline S3: ION2MASS hard table replaced)')

log('adduct parser ready')


In [ ]:
# ===== S7: hill_str + enum_formulas, 7GR-filtered (VERBATIM from gen-p2 v8) =====
# S7: formula enumeration, 7GR-filtered, Hill-order strings (identical to gen-p2 S4)
def hill_str(d):
    parts = []
    if d.get('C'):
        parts.append('C' + (str(d['C']) if d['C'] > 1 else ''))
    if d.get('H'):
        parts.append('H' + (str(d['H']) if d['H'] > 1 else ''))
    for e in sorted(k for k in d if k not in ('C', 'H') and d[k]):
        parts.append(e + (str(d[e]) if d[e] > 1 else ''))
    return ''.join(parts)

def enum_formulas(target, tol, halogens=False, cap=400):
    lo, hi = target - tol, target + tol
    out = []
    n_max = int(hi / EMASS['N']) + 1
    o_max = int(hi / EMASS['O']) + 1
    s_max = min(int(hi / EMASS['S']) + 1, 4)
    p_max = min(int(hi / EMASS['P']) + 1, 3)
    f_max = min(int(hi / EMASS['F']) + 1, 9) if halogens else 1
    cl_max = min(int(hi / EMASS['Cl']) + 1, 5) if halogens else 1
    br_max = min(int(hi / EMASS['Br']) + 1, 3) if halogens else 1
    for n in range(n_max):
        m_n = n * EMASS['N']
        for o in range(o_max):
            m_no = m_n + o * EMASS['O']
            if m_no > hi:
                break
            for s in range(s_max):
                m_nos = m_no + s * EMASS['S']
                if m_nos > hi:
                    break
                for p in range(p_max):
                    m_nosp = m_nos + p * EMASS['P']
                    if m_nosp > hi:
                        break
                    for f in range(f_max):
                        m1 = m_nosp + f * EMASS['F']
                        if m1 > hi:
                            break
                        for cl in range(cl_max):
                            m2 = m1 + cl * EMASS['Cl']
                            if m2 > hi:
                                break
                            for br in range(br_max):
                                m3 = m2 + br * EMASS['Br']
                                if m3 > hi:
                                    break
                                rem_lo, rem_hi = lo - m3, hi - m3
                                if rem_hi < 12.0:
                                    continue
                                c_hi = int(rem_hi / 12.0)
                                c_lo = max(1, int(rem_lo / (12.0 + 3.1 * EMASS['H'])) - 1)
                                for c in range(c_lo, c_hi + 1):
                                    h = round((target - m3 - 12.0 * c) / EMASS['H'])
                                    if h < 0:
                                        continue
                                    mass = m3 + 12.0 * c + h * EMASS['H']
                                    if not (lo <= mass <= hi):
                                        continue
                                    x = f + cl + br
                                    dbe = (2 * c + 2 + n - h - x) / 2.0
                                    if dbe < 0 or dbe != int(dbe):
                                        continue
                                    if (int(round(mass)) % 2 == 0) != (n % 2 == 0):
                                        continue
                                    if not (0.2 <= h / c <= 3.1):
                                        continue
                                    if n / c > 1.3 or o / c > 1.2 or p / c > 0.35 or s / c > 0.8:
                                        continue
                                    if x / c > 1.5 or (n + o + p + s) / c > 2.0:
                                        continue
                                    if dbe > 0.67 * c + 1:
                                        continue
                                    d = dict(C=c, H=h, N=n, O=o, S=s, P=p, F=f, Cl=cl, Br=br)
                                    out.append((hill_str(d), mass))
                                    if len(out) >= cap:
                                        return out
    return out
log('formula enumeration ready')


In [ ]:
# ===== S8: pool load + P_MASS/P_SMI/P_FORM/P_NPL + pool_window (VERBATIM) =====
# S8: candidate source 1 -- pool retrieval by neutral mass window (identical to gen-p2 S5)
pool = pd.read_parquet(POOL_PQ)
keep_cols = ['smiles', 'formula', 'mass'] + (['np_likeness'] if 'np_likeness' in pool.columns else [])
pool = pool[keep_cols].dropna(subset=['smiles', 'mass'])
order = np.argsort(pool['mass'].values)
P_MASS = pool['mass'].values[order]
P_SMI = pool['smiles'].values[order]
P_FORM = pool['formula'].values[order]
P_NPL = pool['np_likeness'].values[order] if 'np_likeness' in pool.columns else np.zeros(len(pool))
print('pool rows:', len(P_MASS))

def pool_window(mass, ppm):
    w = mass * ppm * 1e-6
    lo = np.searchsorted(P_MASS, mass - w, 'left')
    hi = np.searchsorted(P_MASS, mass + w, 'right')
    return np.arange(lo, hi)
log('pool arrays ready: %d rows' % len(P_MASS))


In [ ]:
# ===== S9: Gen-A transforms + derive_products (VERBATIM from gen-p2 v8) =====
# S9: Gen-A derivation (identical to gen-p2 S7)
if not RDKIT_OK:
    print('Gen-A skipped: rdkit unavailable')
    def derive_products(*a, **k):
        return []
else:
    TRANSFORMS = [
        ('[c;H1:1]>>[c:1]O', dict(O=1)),
        ('[C;H2,H3;!$(C=O):1]>>[C:1]O', dict(O=1)),
        ('[C;H1;!$(C=O):1]>>[C:1]O', dict(O=1)),
        ('[O;H1;!$(OC=O):1]>>[O:1]C', dict(C=1, H=2)),
        ('[N;H1,H2:1]>>[N:1]C', dict(C=1, H=2)),
        ('[c;H1:1]>>[c:1]C', dict(C=1, H=2)),
        ('[c;H1:1]>>[c:1]OC', dict(C=1, H=2, O=1)),
        ('[O;H1;!$(OC=O):1]>>[O:1]C(C)=O', dict(C=2, H=2, O=1)),
        ('[N;H1,H2:1]>>[N:1]C(C)=O', dict(C=2, H=2, O=1)),
        ('[O;H1;!$(OC=O):1]>>[O:1]C1OC(CO)C(O)C(O)C1O', dict(C=6, H=10, O=5)),
        ('[O;H1;!$(OC=O):1]>>[O:1]C1OCC(O)C(O)C1O', dict(C=5, H=8, O=4)),
        ('[O;H1;!$(OC=O):1]>>[O:1]C1OC(C)C(O)C(O)C1O', dict(C=6, H=10, O=4)),
        ('[c;H1:1]>>[c:1]CC=C(C)C', dict(C=5, H=8)),
        ('[O;H1;!$(OC=O):1]>>[O:1]CC=C(C)C', dict(C=5, H=8)),
        ('[C:1]=[C:2]>>[C:1][C:2]', dict(H=2)),
        ('[C;H1,H2:1][C;H1,H2:2]>>[C:1]=[C:2]', dict(H=-2)),
        ('[C;H1:1][O;H1:2]>>[C:1]=[O:2]', dict(H=-2)),
        ('[c;H1:1]>>[c:1]C(=O)O', dict(C=1, O=2)),
        ('[c;H1:1]>>[c:1]N', dict(N=1, H=1)),
        ('[O:1][CH3]>>[O:1]', dict(C=-1, H=-2)),
        ('[N:1][CH3]>>[N:1]', dict(C=-1, H=-2)),
        ('[c:1][OH]>>[c:1]', dict(O=-1)),
        ('[C;!$(C=O):1][OH]>>[C:1]', dict(O=-1)),
        ('[O:1]C(=O)[CH3]>>[O:1]', dict(C=-2, H=-2, O=-1)),
        ('[O:1]C1OC(CO)C(O)C(O)C1O>>[O:1]', dict(C=-6, H=-10, O=-5)),
        ('[c:1]CC=C(C)C>>[c:1]', dict(C=-5, H=-8)),
        ('[C,c:1]C(=O)[OH]>>[C,c:1]', dict(C=-1, O=-2)),
    ]
    def delta_mass(d):
        return float(sum(EMASS[e] * n for e, n in d.items()))

    def derive_products(parent_smi, target_mass, tol, max_steps=2, cap=60):
        pm = Chem.MolFromSmiles(parent_smi)
        if pm is None:
            return []
        try:
            pmass = rdMolDescriptors.CalcExactMolWt(pm)
        except Exception:
            return []
        delta = target_mass - pmass
        if abs(delta) <= tol:
            return []
        rxns = []
        for smarts, dd in TRANSFORMS:
            dm = delta_mass(dd)
            if dm == 0 or np.sign(dm) != np.sign(delta):
                continue
            try:
                rxns.append((rdChemReactions.ReactionFromSmarts(smarts), dm))
            except Exception:
                pass
        seen = {Chem.MolToSmiles(pm)}
        hits = {}
        frontier = [(pm, 0.0)]
        for step in range(max_steps):
            nxt = []
            for mol, acc in frontier:
                for rxn, dm in rxns:
                    if abs(acc + dm) > abs(delta) + tol:
                        continue
                    try:
                        psets = rxn.RunReactants((mol,))
                    except Exception:
                        continue
                    for pset in psets[:8]:
                        p = pset[0]
                        try:
                            Chem.SanitizeMol(p)
                        except Exception:
                            continue
                        cs = Chem.MolToSmiles(p)
                        if cs in seen:
                            continue
                        seen.add(cs)
                        acc2 = acc + dm
                        if abs(delta - acc2) <= tol:
                            try:
                                m2 = rdMolDescriptors.CalcExactMolWt(p)
                            except Exception:
                                continue
                            if abs(m2 - target_mass) <= tol:
                                hits[cs] = p
                        elif step + 1 < max_steps and abs(acc2) < abs(delta):
                            nxt.append((p, acc2))
                        if len(hits) >= cap:
                            break
                    if len(hits) >= cap:
                        break
                if len(hits) >= cap:
                    break
            frontier = nxt
        out = []
        for cs, p in hits.items():
            try:
                out.append(dict(smiles=cs, formula=rdMolDescriptors.CalcMolFormula(p),
                                mass=rdMolDescriptors.CalcExactMolWt(p), npl=0.0, src='gen'))
            except Exception:
                pass
        return out
log('Gen-A ready: %d reaction SMARTS' % len(TRANSFORMS))


In [ ]:
# ===== M-S13-A: sig_ms2 — MS2 fragment / neutral-loss consistency, 100% train-library-free =====
# inputs: the QUERY spectra (test.parquet) and the CANDIDATE structure only.
#   (a) cut every single bond between two heavy atoms -> each piece contributes '|<exact mass>'
#       and its complement '^<loss>'; three retro-Diels-Alder SMARTS add fragments at half weight;
#   (b) 30 hand-written neutral-loss motifs tested as substructures of the candidate.
# score = intensity-weighted fraction of observed peak mass the candidate explains
#         + 0.5 * fraction of the observed motif losses the candidate can produce.
RETRO_DIELS = ['[#6]=[#6]-[#6]-[#6]', '[#6]=[#6]-[#6]~[#6]', '[#6]=[#6]-[#6]']
NL_BASE = [(18.010565, 'O'), (17.026549, '[NH2,NH3+]'), (15.994915, ''), (28.031300, '[CH]=O'),
           (26.015650, ''), (27.994915, ''), (42.010565, 'C(=O)'), (29.039125, ''),
           (31.989829, ''), (32.026215, ''), (38.968853, 'S'), (43.018390, ''),
           (43.989829, ''), (44.026215, ''), (45.021465, ''), (46.005480, 'S=O'),
           (46.041865, ''), (57.021464, ''), (58.005480, ''), (60.021129, ''),
           (62.036779, ''), (72.021129, ''), (74.036779, ''), (86.036779, ''),
           (90.031694, 'C=CC'), (102.031694, ''), (132.042259, 'OC1OCC(O)C(O)C1'),
           (146.057909, ''), (162.052824, 'OC1OC(CO)C(O)C(O)C1'), (176.032088, '')]
MAX_FRAG = 400
FRAG_CACHE = {}
SIG_CACHE = {}
Q_SPECS = {}

try:
    NL_PATTERNS_SIG = [(lm, Chem.MolFromSmarts(sma) if sma else None) for lm, sma in NL_BASE]
except Exception:
    traceback.print_exc()
    NL_PATTERNS_SIG = [(lm, None) for lm, sma in NL_BASE]

if RDKIT_OK:
    NL_RULES = [
        (18.010565, '[OH]'),
        (27.994915, '[CH]=O'),
        (43.989829, 'C(=O)[OH]'),
        (17.026549, '[NH2,NH3+]'),
        (162.052824, 'OC1OC(CO)C(O)C(O)C1O'),
        (132.042259, 'OC1OCC(O)C(O)C1O'),
        (146.057909, 'OC1OC(C)C(O)C(O)C1O'),
        (176.032088, 'OC1OC(C(=O)O)C(O)C(O)C1O'),
        (79.956815, 'OS(=O)(=O)O'),
        (86.000394, 'OC(=O)CC(=O)O'),
    ]
    NL_PATTERNS = [(lm, Chem.MolFromSmarts(sma)) for lm, sma in NL_RULES]
else:
    NL_RULES = []
    NL_PATTERNS = []

_MOL_CACHE = {}


def get_mol(smi):
    mo = _MOL_CACHE.get(smi, 0)
    if mo == 0:
        mo = Chem.MolFromSmiles(smi) if RDKIT_OK else None
        _MOL_CACHE[smi] = mo
    return mo


def observed_losses(spectra, tol=0.02, max_peaks=30):
    observed = set()
    for s in spectra:
        prec = s['prec']
        for pk in s['mz'][:max_peaks]:
            loss = prec - pk
            for i, (lm, _) in enumerate(NL_PATTERNS):
                if abs(loss - lm) <= tol:
                    observed.add(i)
    return observed


def nl_score(smi, observed):
    if not RDKIT_OK or not observed:
        return 0.0
    mol = get_mol(smi)
    if mol is None:
        return 0.0
    sc = 0.0
    for i in observed:
        patt = NL_PATTERNS[i][1]
        if patt is not None and mol.HasSubstructMatch(patt):
            sc += 1.0
    return sc


def _frag_smiles(mol):
    out = []
    try:
        bonds = [b.GetIdx() for b in mol.GetBonds()
                 if b.GetBondType() == Chem.BondType.SINGLE
                 and b.GetBeginAtom().GetAtomicNum() > 1 and b.GetEndAtom().GetAtomicNum() > 1]
    except Exception:
        return out
    for i in bonds:
        try:
            e = Chem.FragmentOnBonds(mol, [i], addDummies=True)
            for f in Chem.GetMolFrags(e, asMols=True, sanitizeFrags=False):
                try:
                    f2 = Chem.RemoveHs(f)
                    rw = Chem.RWMol()
                    for a in f2.GetAtoms():
                        if a.GetSymbol() != '*':
                            rw.AddAtom(Chem.Atom(a.GetAtomicNum()))
                    pos = {old: new for new, old in enumerate(a.GetIdx() for a in f2.GetAtoms()
                                                              if a.GetSymbol() != '*')}
                    for b in f2.GetBonds():
                        if b.GetBeginAtomIdx() in pos and b.GetEndAtomIdx() in pos:
                            rw.AddBond(pos[b.GetBeginAtomIdx()], pos[b.GetEndAtomIdx()], b.GetBondType())
                    m3 = rw.GetMol()
                    if m3.GetNumAtoms() == 0:
                        continue
                    Chem.SanitizeMol(m3)
                    out.append(Chem.MolToSmiles(m3))
                except Exception:
                    continue
        except Exception:
            continue
        if len(out) >= MAX_FRAG:
            break
    return out


def _retro_diel_frags(mol):
    out = []
    for sma in RETRO_DIELS:
        try:
            patt = Chem.MolFromSmarts(sma)
            if patt is None:
                continue
            for match in mol.GetSubstructMatches(patt):
                if len(match) < 2:
                    continue
                for keep in (match[:len(match) // 2], match[len(match) // 2:]):
                    try:
                        rw = Chem.RWMol(mol)
                        for i in sorted(set(match) - set(keep), reverse=True):
                            rw.RemoveAtom(int(i))
                        m2 = rw.GetMol()
                        Chem.SanitizeMol(m2)
                        for f in Chem.GetMolFrags(m2, asMols=True):
                            if f.GetNumAtoms():
                                out.append(Chem.MolToSmiles(f))
                    except Exception:
                        continue
                if len(out) >= 2 * MAX_FRAG:
                    return out
        except Exception:
            continue
    return out


def explanation(smi, mol):
    if smi in FRAG_CACHE:
        return FRAG_CACHE[smi]
    acc = collections.defaultdict(float)
    if mol is None:
        FRAG_CACHE[smi] = acc
        return acc
    try:
        mw = rdMolDescriptors.CalcExactMolWt(mol)
    except Exception:
        FRAG_CACHE[smi] = acc
        return acc
    for s2 in _frag_smiles(mol):
        try:
            m2 = Chem.MolFromSmiles(s2)
            if m2 is None:
                continue
            mm = rdMolDescriptors.CalcExactMolWt(m2)
        except Exception:
            continue
        acc['|%.3f' % mm] += 1.0
        acc['^%.3f' % (mw - mm)] += 1.0
    for s2 in _retro_diel_frags(mol):
        try:
            m2 = Chem.MolFromSmiles(s2)
            if m2 is None:
                continue
            mm = rdMolDescriptors.CalcExactMolWt(m2)
        except Exception:
            continue
        acc['|%.3f' % mm] += 0.5
        acc['^%.3f' % (mw - mm)] += 0.5
    FRAG_CACHE[smi] = acc
    return acc


def cum_spectrum(mz, it, tol=0.01):
    if len(mz) == 0:
        return {}
    acc = collections.defaultdict(float)
    prev, prev_key = None, None
    for j in range(len(mz)):
        if prev is not None and mz[j] - prev <= tol:
            acc[prev_key] += float(it[j])
            continue
        prev = mz[j]
        prev_key = '|%.3f' % mz[j]
        acc[prev_key] += float(it[j])
    tot = sum(acc.values()) or 1.0
    return {k: v / tot for k, v in acc.items()}


def observed_loss_ids(specs, tol=0.01):
    keys = set()
    for s in specs:
        prec, mz = s['prec'], s['mz']
        if not len(mz):
            continue
        L = prec - mz
        for pid, (lm, _) in enumerate(NL_BASE):
            if np.any(np.abs(L - lm) <= tol):
                keys.add(pid)
    return keys


def loss_match(mol, obs):
    if not obs or mol is None:
        return 0.0
    got = 0
    for pid in obs:
        patt = NL_PATTERNS_SIG[pid][1] if pid < len(NL_PATTERNS_SIG) else None
        if patt is None:
            continue
        try:
            if mol.HasSubstructMatch(patt):
                got += 1
        except Exception:
            continue
    return got / max(1, len(obs))


def sig_ms2(mid, smi, specs=None):
    """cached (molecule, candidate) MS2 consistency in [0, 1.5]; 0.0 when rdkit is unavailable."""
    c = SIG_CACHE.get(mid)
    if c is None:
        c = SIG_CACHE[mid] = {}
    if smi in c:
        return c[smi]
    v = 0.0
    if RDKIT_OK and smi not in ('', None):
        try:
            specs = specs if specs else (Q_SPECS.get(mid) or [])
            cums = [cum_spectrum(s['mz'], s['it']) for s in specs]
            obs = observed_loss_ids(specs)
            mol = get_mol(smi)
            if mol is not None and cums:
                expl = explanation(smi, mol)
                hit = 0.0
                for cum in cums:
                    for k, w in cum.items():
                        vv = expl.get(k)
                        if vv is None:
                            continue
                        hit += w * (1.0 - math.exp(-vv))
                v = hit / max(1, len(cums)) + 0.5 * loss_match(mol, obs)
        except Exception:
            v = 0.0
    c[smi] = v
    return v


log('sig_ms2 primitives ready (%d neutral-loss motifs, %d base NL rules)' % (
    len(NL_BASE), len(NL_RULES)))


In [ ]:
# ===== M-S2 (VERBATIM): load test spectra, clean peaks (mz <= prec + 2.0 KEPT), group by id =====
# S2: load test spectra, clean peaks, group by molecule_id
test = pd.read_parquet(TEST_PQ)
print(test.shape, list(test.columns))

def as_float_list(x):
    if x is None:
        return []
    if isinstance(x, str):
        try:
            x = json.loads(x)
        except Exception:
            return []
    try:
        return [float(v) for v in x]
    except Exception:
        return []

def clean_peaks(mzs, its, prec, max_peaks=160, floor=1e-3):
    mz = np.asarray(mzs, dtype=np.float64)
    it = np.asarray(its, dtype=np.float64)
    keep = (mz <= prec + 2.0) & (mz > 0)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0 or it.max() <= 0:
        return np.zeros(0), np.zeros(0)
    it = it / it.max()
    keep = it >= floor
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        sel = np.argsort(-it)[:max_peaks]
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    return mz[o], it[o]

mols = {}
for r in test.itertuples(index=False):
    mid = str(r.molecule_id)
    mz, it = clean_peaks(as_float_list(r.ms2_mzs), as_float_list(r.ms2_normalized_intensities), float(r.precursor_mz))
    mols.setdefault(mid, dict(spectra=[]))['spectra'].append(dict(
        mz=mz, it=it, prec=float(r.precursor_mz),
        adduct=getattr(r, 'adduct', None), mode=getattr(r, 'ionization_mode', None)))
log('M-S2 molecules %d | spectra %d' % (len(mols), sum(len(v['spectra']) for v in mols.values())))


In [ ]:
# ===== M-S3: adduct parsing -> neutral monoisotopic mass (VERBATIM gen-p2 v8 M-S3) =====
for mid, m in mols.items():
    ests = [neutral_mass_parser(s['prec'], s['adduct'], s['mode']) for s in m['spectra']]
    ests = [e for e in ests if np.isfinite(e) and 50.0 < e < 3000.0]
    m['mass'] = float(np.median(ests)) if ests else float('nan')
ok_mass = [m['mass'] for m in mols.values() if np.isfinite(m['mass'])]
n_known = sum(1 for m in mols.values()
              for s in m['spectra'] if parse_adduct(s['adduct']) and parse_adduct(s['adduct'])['known'])
n_spec = sum(len(m['spectra']) for m in mols.values())
log('M-S3 neutral mass median %s | no-mass mols %d | known-parse spectra %d/%d' % (
    float(np.median(ok_mass)) if ok_mass else 'NA',
    sum(1 for m in mols.values() if not np.isfinite(m['mass'])), n_known, n_spec))


In [ ]:
# ===== M-S4: formula enumeration, 7GR-filtered (VERBATIM gen-p2 v8 M-S4) =====
for mid, m in mols.items():
    if not np.isfinite(m['mass']):
        m['formulas'], m['formula_set'] = [], set()
        continue
    fs = enum_formulas(m['mass'], CFG['FORMULA_TOL_DA'], CFG['HALOGENS'], CFG['FORMULA_CAP'])
    m['formulas'] = fs
    m['formula_set'] = {f[0] for f in fs}
log('M-S4 formulas/molecule median %d' % int(np.median([len(m['formulas']) for m in mols.values()])))


In [ ]:
# ===== M-S5: +/-10 ppm pool retrieval (VERBATIM gen-p2 v8 M-S5; NO np_likeness cap) =====
for mid, m in mols.items():
    if not np.isfinite(m['mass']):
        m['cands'] = []
        continue
    idx = pool_window(m['mass'], CFG['MASS_PPM'])
    if len(idx) == 0:
        idx = pool_window(m['mass'], CFG['MASS_PPM_FALLBACK'])
    cands, seen = [], set()
    for i in idx:
        smi = P_SMI[i]
        if smi in seen:
            continue
        seen.add(smi)
        cands.append(dict(smiles=smi, formula=str(P_FORM[i]), mass=float(P_MASS[i]),
                          npl=float(P_NPL[i]), src='pool'))
    m['cands'] = cands
log('M-S5 pool candidates/molecule median %d' % int(np.median([len(m['cands']) for m in mols.values()])))


In [ ]:
# ===== M-S7: Gen-A derivation (VERBATIM gen-p2 v8 M-S7; derive_products reused) =====
if RDKIT_OK and CFG['GEN_DERIVE']:
    n_gen = 0
    for mid, m in mols.items():
        if not np.isfinite(m['mass']) or not m['cands']:
            continue
        parents = sorted(m['cands'], key=lambda c: -c['npl'])[:CFG['GEN_PARENTS']]
        have = {c['smiles'] for c in m['cands']}
        for pc in parents:
            try:
                prods = derive_products(pc['smiles'], m['mass'], CFG['GEN_TOL_DA'], 2,
                                        CFG['GEN_CAP_PARENT'])
            except Exception:
                traceback.print_exc()
                continue
            for g in prods:
                if g['smiles'] in have:
                    continue
                if m['formula_set'] and g['formula'] not in m['formula_set']:
                    continue
                have.add(g['smiles'])
                m['cands'].append(g)
                n_gen += 1
    log('M-S7 generated candidates added: %d' % n_gen)
else:
    log('M-S7 Gen-A disabled (rdkit=%s, GEN_DERIVE=%s)' % (RDKIT_OK, CFG['GEN_DERIVE']))


In [ ]:
# ===== M-S9: base scoring (VERBATIM gen-p2 v8 M-S9) -> STAGE 2 atomic write =====
for mid in sorted(mols):
    m = mols[mid]
    obs = observed_losses(m['spectra'])
    _MOL_CACHE.clear()
    for c in m['cands']:
        c['score'] = (nl_score(c['smiles'], obs) + 0.5 * c['npl']
                      + (0.3 if c['src'] != 'pool' else 0.0))
    _MOL_CACHE.clear()
    Q_SPECS[mid] = m['spectra']


def base_rank(m):
    """gen-p2 v8's own base order: sort by -score, keep first occurrence of each SMILES."""
    seen, out = set(), []
    for c in sorted(m['cands'], key=lambda c: -c['score']):
        if c['smiles'] in seen:
            continue
        seen.add(c['smiles'])
        out.append(c)
    return out


def fallback_smis(top_k):
    fb = [p for p in sorted(zip(P_SMI, P_NPL), key=lambda x: -x[1])[:top_k * 4]]
    out = []
    for smi, _ in fb:
        if smi not in out:
            out.append(smi)
        if len(out) >= top_k:
            break
    return out


FALLBACK = fallback_smis(CFG['TOP_K_SUBMIT'])


def order_to_25(smis, order, fallback=None):
    ranked = [smis[i] for i in order][:CFG['TOP_K_SUBMIT']]
    if len(ranked) < CFG['TOP_K_SUBMIT']:
        ranked = ranked + [s for s in (fallback or FALLBACK) if s not in ranked]
    return ranked[:CFG['TOP_K_SUBMIT']]


BASE_LISTS = {}
for mid in sorted(mols):
    BASE_LISTS[mid] = order_to_25([c['smiles'] for c in base_rank(mols[mid])],
                                  list(range(len(mols[mid]['cands']))))
write_sub([(mid, BASE_LISTS[mid]) for mid in sorted(mols)], 'stage2_base')
log('STAGE 2 (M-S9 base scoring) written')


In [ ]:
# ===== NATIVE POOL: the exact gen-p2 v8 `ice_candidates(base_rank(m), 60)` pool =====
# This pool is shared by GLACIER and ICEBERG. NO base_score / sig_ms2 / union-floor pruning.
try:
    sys.path.insert(0, ICE_PKG)
    from fuse import build_ice_input, ice_candidates
except Exception:
    traceback.print_exc()
    raise

cands_by_mid = {}
for mid in sorted(mols):
    base = base_rank(mols[mid])
    cands_by_mid[mid] = ice_candidates([c['smiles'] for c in base],
                                       [c['formula'] for c in base], CFG['TOP_N_RESCORE'])
N_CANDS_NATIVE = int(sum(len(v) for v in cands_by_mid.values()))
N_MOLS_NATIVE = int(sum(1 for v in cands_by_mid.values() if v))
log('native pool: %d candidates | %d molecules with candidates' % (N_CANDS_NATIVE, N_MOLS_NATIVE))

# build the single `items` object with FULL metadata columns (R44 bug fix)
test_ice = test.copy()
test_ice['ms2_mzs'] = test_ice['ms2_mzs'].map(as_float_list)
test_ice['ms2_normalized_intensities'] = test_ice['ms2_normalized_intensities'].map(as_float_list)
items = build_ice_input(test_ice, cands_by_mid)
N_ITEMS = int(len(items))
log('build_ice_input: %d items | test_ice columns %r' % (N_ITEMS, list(test_ice.columns)))


In [ ]:
# ===== multi-process runner for ICEBERG / GLACIER (N_WORKERS=2, 2 threads each) =====
from concurrent.futures import ProcessPoolExecutor, as_completed


def _set_worker_env(workdir, n_threads):
    import os, torch
    os.environ['OMP_NUM_THREADS'] = str(n_threads)
    os.environ['MKL_NUM_THREADS'] = str(n_threads)
    os.environ['OPENBLAS_NUM_THREADS'] = str(n_threads)
    os.environ['NUMEXPR_NUM_THREADS'] = str(n_threads)
    os.environ['TMPDIR'] = workdir
    os.environ['TEMP'] = workdir
    os.environ['TMP'] = workdir
    torch.set_num_threads(n_threads)
    os.makedirs(workdir, exist_ok=True)


def _mid_of(item):
    if isinstance(item, dict):
        return str(item['molecule_id'])
    return str(item[0])


def _split_by_molecule(items, n_workers):
    by_mid = collections.OrderedDict()
    for it in items:
        by_mid.setdefault(_mid_of(it), []).append(it)
    mids = list(by_mid.keys())
    slices = [[] for _ in range(n_workers)]
    for i, mid in enumerate(mids):
        slices[i % n_workers].extend(by_mid[mid])
    return slices, mids


def _count_real_scores(scores):
    """inner floating-point score counts; rejects outer-dict-non-empty false positives."""
    n_scored, n_mols_scored = 0, 0
    for d in scores.values():
        if isinstance(d, dict):
            finite = [v for v in d.values() if isinstance(v, (int, float)) and np.isfinite(v)]
            n_scored += len(finite)
            if finite:
                n_mols_scored += 1
    return n_scored, n_mols_scored


def _ice_worker(payload):
    idx, items_slice, ice_pkg, workdir, site, budget_s = payload
    _set_worker_env(workdir, OMP_PER_WORKER)
    import sys
    sys.path.insert(0, ice_pkg)
    from fuse import run_ice
    return run_ice(ice_pkg, items_slice, workdir=workdir, device='cpu',
                   budget_s=budget_s, site=site)


def _gl_worker(payload):
    idx, items_slice, gl_pkg, workdir, site, budget_s, ice_pkg = payload
    _set_worker_env(workdir, OMP_PER_WORKER)
    import sys, os
    sys.path.insert(0, gl_pkg)
    from gl_fuse import run_gl
    return run_gl(gl_pkg, items_slice, workdir=workdir, device='cpu',
                  budget_s=budget_s, site=site, wheels=os.path.join(ice_pkg, 'wheels'))


def _merge_scores(results):
    merged = {}
    for r in results:
        if isinstance(r, dict):
            merged.update(r)
    return merged


def _run_ice_parallel(items, ice_pkg, gl_pkg, work_root, site, budget_s, n_workers=N_WORKERS):
    # prepare wheel site once in the main process so workers can share it
    try:
        _ice_worker((-1, items[:1] if items else [], ice_pkg, site, site, 30.0))
        log('ice site prepared at %s' % site)
    except Exception:
        traceback.print_exc()
        log('[warn] ice site prep failed; workers may race on wheel install')

    slices, mids = _split_by_molecule(items, n_workers)
    expected_per_worker = max(1, len(mids) // n_workers)
    payloads = [(i, slices[i], ice_pkg, os.path.join(work_root, 'ice_work_%d' % i),
                 site, budget_s / n_workers) for i in range(n_workers)]

    results, exceptions = [], []
    try:
        with ProcessPoolExecutor(max_workers=n_workers) as ex:
            futures = [ex.submit(_ice_worker, p) for p in payloads]
            for f in as_completed(futures):
                try:
                    results.append(f.result())
                except Exception as exc:
                    traceback.print_exc()
                    exceptions.append(str(exc))
                    results.append({})
    except Exception:
        traceback.print_exc()
        exceptions.append('ProcessPoolExecutor failed')

    merged = _merge_scores(results)
    n_scored, n_mols_scored = _count_real_scores(merged)
    log('ICE parallel preliminary: %d molecules covered | %d scored | %d finite scores' % (
        len(merged), n_mols_scored, n_scored))

    # fallback: any molecule with no real inner score -> single-process rescue
    missing = [m for m in mids if m not in merged
               or not any(isinstance(v, (int, float)) and np.isfinite(v)
                          for v in merged.get(m, {}).values())]
    if exceptions or len(missing) > max(1, expected_per_worker // 10):
        log('ICE fallback: exceptions=%d missing=%d/%d, running single process rescue' % (
            len(exceptions), len(missing), len(mids)))
        try:
            missing_items = [it for it in items if _mid_of(it) in missing]
            fb = _ice_worker((-1, missing_items, ice_pkg,
                              os.path.join(work_root, 'ice_fallback'), site, budget_s))
            if isinstance(fb, dict):
                for m in missing:
                    if m in fb:
                        merged[m] = fb[m]
        except Exception:
            traceback.print_exc()

    return merged


def _run_gl_parallel(items, gl_pkg, ice_pkg, work_root, site, budget_s, n_workers=N_WORKERS):
    slices, mids = _split_by_molecule(items, n_workers)
    expected_per_worker = max(1, len(mids) // n_workers)
    payloads = [(i, slices[i], gl_pkg, os.path.join(work_root, 'gl_work_%d' % i),
                 site, budget_s / n_workers, ice_pkg) for i in range(n_workers)]

    results, exceptions = [], []
    try:
        with ProcessPoolExecutor(max_workers=n_workers) as ex:
            futures = [ex.submit(_gl_worker, p) for p in payloads]
            for f in as_completed(futures):
                try:
                    results.append(f.result())
                except Exception as exc:
                    traceback.print_exc()
                    exceptions.append(str(exc))
                    results.append({})
    except Exception:
        traceback.print_exc()
        exceptions.append('ProcessPoolExecutor failed')

    merged = _merge_scores(results)
    n_scored, n_mols_scored = _count_real_scores(merged)
    log('GL parallel preliminary: %d molecules covered | %d scored | %d finite scores' % (
        len(merged), n_mols_scored, n_scored))

    missing = [m for m in mids if m not in merged
               or not any(isinstance(v, (int, float)) and np.isfinite(v)
                          for v in merged.get(m, {}).values())]
    if exceptions or len(missing) > max(1, expected_per_worker // 10):
        log('GL fallback: exceptions=%d missing=%d/%d, running single process rescue' % (
            len(exceptions), len(missing), len(mids)))
        try:
            missing_items = [it for it in items if _mid_of(it) in missing]
            fb = _gl_worker((-1, missing_items, gl_pkg,
                             os.path.join(work_root, 'gl_fallback'), site, budget_s, ice_pkg))
            if isinstance(fb, dict):
                for m in missing:
                    if m in fb:
                        merged[m] = fb[m]
        except Exception:
            traceback.print_exc()

    return merged


log('multi-process runner ready: N_WORKERS=%d threads/worker=%d' % (N_WORKERS, OMP_PER_WORKER))


In [ ]:
# ===== in-kernel evaluation: the vis400 reference table (diagnostics only, never a candidate) =====
REF_SHA256 = 'ca145a39507748a0d84dfc42e3c8cc9ea198f3fa497d1a06890404fc0fb33774'
REF_B64 = (
    'eNqdXcuO7Dhy/ZeeTQqYBkRS1GOA3oyAglepxV0ahiGSkhduq4Ap7wbz7+abEXxkKt24jXurEKQkPoIRJ04E//nb//xn3/ODs9/+'
    '9s/f9uvnt7/9tj4fqySX1P+R9fHH1m3dupKHpPoXj7//o5OSml/89tff/ut/PxL/x3//9rf5X3+1jxwJeuQmiRF/0nV92E6ekl2X'
    'vNjK1pXp9lT/Tj8gPNT31vveBN/3vLdLPlbT36Y7eKyr7YZ2kmxrfPP3cuYhxD2EHAul4CHujd0X6/czLU078/O/X//2H41XpXxR'
    'Q+pl+2N9PHXDy74I6fS/db/f8vG7HT8zdlerJ0XmGbyPHzY3jv4nPZhd7Ei/7J9dawgZmQXP3sv19WXGvVsf9ifzzyfRL/y4TLcX'
    'Wgm4x4Ed8wh61O/ytOvEvoYkX91qetqaHfBenAR+oG6tR8a0sB9n10qn/2x6kZjZg2N1W9rO8OCfKDg9ylXp3vd384MeWdObGVOz'
    'bjo9AGaywXM/a2OeTql7+jgzPqAJdQ1XPUqXbUxXMxH6N2Eu6gM3HoQpPHDf9smSPH+ZHu3rkPCWfg3r36xw9O43MY+e/KNPxha8'
    'SX7CANhdZmbhYm4cJFvNVmst8KlfZg4/w0yg/l9PJnylsKLA1L8XNI8Z/WOGY4aqyM9g0GZ2ta/g3w+j8PSy74AK/KwNUCrzQsep'
    '3HbXpTWR3nn6j95+fsutUU9R8OzPmgAVPKsJbi/fj1a6VnvhyY7dZk99L20eyN0D91mccD6fZohWPTJpkqyqYtIuPL3cWdfSDfvJ'
    'yY7WGdaZeiLQsrVT82ecr9vidqr8XMnhID3an6v/7qBgrIZ5MneWdPY4kHq7Qq30URurHfzD5XLseLbsi1I3A/pvN+UbPHfxKrkp'
    'Dx6q+lkAffgEC/q5uoZfHZxxdBzcFwcbQlGygA3hTowr7CuzYB7PxxrtDIo3w31xoLMU47wvtf62mne1+9Zqr8ez8yfI2jxF1TCy'
    's7an3GL70V/847empN+ymKI7wnZ+/JY69nln2ZFBzJZyo73+5dn5M7oLK699bBzHMIp8T/mBpA+zLc3mNKPoxkCP8lbv6dQW5VSM'
    'qBnNeBJas0JSqFPei4GleR6LWrLNaL/w+oEHruuB5lvwnWRaHaSfTyXKb6HGQAy2qlP3Xgkb5bvS6sAQugz8yK0/19pNzmW6DdP0'
    'Ve+EEU7hG2nj8bIKy7yR7s1/kx/LcFA3+hrUUFMr/vs2Gs+yy75eo5uBj9NSqEazlPWW0/1oRWf3kNMC37T7zlXiO1k79U5HaM9B'
    'zwp8aae7nb1Kpf6Czr07NMW+0j67K26eufhHjqzn2K4xr+vn3+wHsz+sWWd2iFfuevNdyajCQ8bVLobiNHw+0tLXFpLWBCyqZ2Ys'
    'R+h3fdDCalj/5JGJU6FvueKe83ra6Jqopi8wWfdk7dPm8DS198i5dAtdnz/o9X9sR05BN059MmlPT1bsc3OO6n6ZXUTDug7dTzhK'
    'S8P8jTCY9IlQtPXhzjIulrWsqDU2u6u1NaZpElAXRjtjXcPRGObPa1izil4MwkzYTjI/1FrTwdYz70PNQf9wzvP1TZv7fz4pPG43'
    'cHro6XgG+8DrRvBOtySBd0N2bWgv+arzJlhwENzSNSqfmfWrlTHypD9sY1ch90+fBUUKyuqzADVQ548436yp5AThEzpsibXpzTT6'
    '7WAXxu/BM24dtETQ6RjLZWw+LB1JfjF8da6nVZ/l+VK+0wCcZOLYD54p6YhirJ0fT/32l/E0jenQ+gA1CYUczM2bSPZkfUIoIjjB'
    'st3ZQWeRr+dntI8ttkIdtPJ8vdXOeUTj6sw/92rJgNH63XtE3sr6RYzEH1s6kT5uZ17CYQi0F3pPZaP8RPhY/KluPdFeMgScOIjo'
    'cRkP1mjNoGv1+tvs+mtoCkqGfujzjhwsEjaR3zLW+ngYhM3BPV/IKL3fJhnylEyCLNlkFL4e06cv7fz2Me6Ptl4a3yLVMJSGynUl'
    'V9dsY+BdeDVauED3mtgvYe7ZbGSyGEePcObujDPknJq6nB74aUwPO88pU+RGD4W1b9W4xRTd8v+BevC9JJgKPpwcTgVQfau1t9aw'
    'xtvaj/KdzUfFprffaa2OK2A68qeGs76VtW+8+KfJEx1ym0cOzUKDfmVAXtvnJR3FmcFqHhkFStvtKz2MwCp9LQV2/MTGZawszYCV'
    'AgvNWZcYNf1APoEpdFInVZma0fP6+JXgTjMqVzigJfmjoW4WdsiKlpCh5Xd4q1cfcLtFAt7o3h9kKj7hMvvz15r8LKsq6/4U3fcZ'
    'oiKmi2CdOe29xuE0EEvA9i2ebfwi6IR82hBssF3SQxVLjKy/rFcYbZawmKxVZnEn5AZ90iZ5wlRwOuw1d9D5B5tx1zFK8loEfJeY'
    'OGF5yMPaFkHxJGjF/HJDKuq9aDJO9Po85wnpKOcQrg6ds4cfNQeO7sM62fYfNG3YDxqANSgpg0ckND4CXO2MDtPtI+I/F/7Quy3A'
    '0CpyiD0/Ho2pQIKP8eOWwGXPEIhf3JAE3jI96MgljgYkWCdMj3EYzNFofFbZ8lnpcarxLA3XX2tYqs/OhumqjldTCozKSQids5DP'
    'jwsLahPTmfrb6k5byV4ED+i57IIUZ5517VNoLKAjZkgJWEk3RJMqZn1PB3QyrhHHAvar/mtDYNcbKWCAMLNW4GL5y4qO1G/n6QWF'
    '5Zza1Tr/DaOb0ZmxfKDNxGMU3G6bFYVW3sqZh/jXHia99rCb4F1+vxKs10I3b0FtTbifDbLvs93izZdnepUEw0n53eyIEV46z26J'
    'RaN/vZixtc0CM5+Ih+CjJknVsFExeVSBgOzM97aYRCr7rjQY/qkf2FjT3gnT1r9y0YDslL0pm3Yum2d2nLl7/4xuYQr9hVHqfmRD'
    'zbB5GchZe3Gz9yIG4zGPoGDtu7WMHLZr1z0LdlkE5hE770KslUZreKWNVbTvw1zFt9ekV03/NEIBDSyX7QfjvBg2j1BZm5A5kzOE'
    '7/UoACDgtridKRqeKRVC0SySEQ5nuKMgFLxudRuMiZ2jdR3wqEeAAfzBEizSYie9lIQvLg8+zTkIG1uWppIBGX+vRv8+bQfgU6ZG'
    'qlTJA3lCuztQDSJmkemPe/JgeykxzUsdRbyA3ZpphMbhyJRckBbcAhaJ3sraBJfb81AV3ZUGr3+OYhcI7Lbv/sus3+CgaNfQv/kF'
    'FNEdSeA4snPukQUR9Vcw7x0NIzNR3knBTzknkS/3GEUw3YT4hVdR18PbKw1tN/S074fMfHDUC3/EGFWi9ZQnTXxnZs8H8gnaHsgk'
    'YcAG0H2c1oj0AQjDtY5nfczMLI9IXcl8ckBVDBZgP+S1HIgk6Kfss6rDGC6Ovm5O8cXAvv6Wq/nKckLO5+rPuojnWhTQuS5ugWc+'
    '1P0GafUYTlRfoODJgrENN7eVrNNiAZkV6/s74tYB8J86DDtHZph5bdCJ0f6RCrFCS/iOZAJChkGKmeZIAjxhzfkNIY4i3PNZEzCu'
    'nOwKfaNbEB2Iz5mlRi/TkzHRwDy+l4QPUiNyidfE0cJhd736ElALRvSmeDIch3GcxwpBKVsHGwiEIBjvA3nAhRvGpVcsW6kSxDsz'
    'ZmHmFuPNNhGEkEX8MgQd0gmiR55d1AEuP63eRjLtJS8sca0ceOMVoB/gy8X40Or+uB1QoNMsS6pgNMY8HJDiojZsQAri6TtpcLIN'
    '036MU/bZFjBDStC8vXn9FNNc0QffbWGfPPknH/0ClZbfjt5+1nOmjbbfE9YHl3LrxJjFeZB8+KxXlVapw6Uvj0tDQBn3JTjfz5pP'
    'l4iflvsD45Nthu4g5lFkYTKnhKAn5L40kgdaxvEghD5pUXzVjhgMzTrH3cbd5NU+Y4XsR2gm/EqrNxHKDKxiBgpC2bibUywTz3An'
    '83gIeHoHSPvK27oNXb5tPmiRYEquxwH5TMFl+rNzTJqwDQ0U6RZXcyw4U+dBaw4iUHF6oJ/durZWIOenOjLkT0/BwznvAPbR820B'
    'YOMb6GOhoZT4OE4HLflIUL0EHBTYOM0Nwsd5R1bhilmDD+tW+/d00LQkf/8HxGnuiSeVxkchkWX7JyBMrI/L+JfOc3eRLQgX3BIF'
    'oCSf6F5gEw+4pPSCsmwayRyhxa0U2ycmXONh07Z5EczwvEezuySJHLgth6yzjk5GRAXSN4iPV3VG023rW13HF0GEyngDwduGSI7r'
    'zpp07aW/CHnk3qcj+wAz2m+gDI5/KQaOfq4PmjHDcqxfZ0Et3WyLJm8kmb04/bk4d4lJinBXBGpjDA68ZilyOZFe4BleV8BEfiJe'
    'ig3aI/PyjnCKS3A5q4Hkj/NcOEcwdcQXd/LJBmLE1U6OswQNkglkzhKHonofugAesw7VTkkJOpscC0+PMVrrF3MUhS6H5G9Kwy17'
    'HidBcQSQEfJwm4l0iXebLMX6F4x9r31Q7DQ66m7YEwRFC7wvYAymNdErP24FkJyxPyYUI8hAbYBZBnV2NaZjJD2XJGdKr5Gh4cwk'
    'GcwkKZsbXPe0L2WMxdBoLmCB05hT4JKV1kZndGA5xcDAbYBg6w2Ri1bIzR/IA3txpMvE8i2K8OstOHVrvjlfiiWfyEBwMy84Mun4'
    'hw6kdy4a480mOg+ZWZ2SObxuD1wzMwOYknZbPAWMxqFnvSyUwe92iUQMma3eQdGj/d0mk40D60f51jsMAPHFjKtrlctbD7HVJoUY'
    'Rr5PZ4H6h5wCGD2P6HQW9sbfMhO0IT0z4nK61o6z44Pb5DFSrJ4bssmTH8U47nt+koIVBJS6M6u/8Gn6ThRAMKOiOES4QkSjYPH9'
    'ON4ugg3uygO8fFRMyT7DQlG0DPQoHc2+ZcSMaqG5+Q7wANvBZalcFQ3yWg5MyUHkgkheziyE4V/LcPRGoIsXNLgh4yHkuaATknjG'
    'V9Bhju4VwpGei7a2w5DjSehUsuW/veVqOXFmWK3SJ942+mod4ePJdi5fmFkJWAQYc9PVHM+BDGXULFumUgaCuE9T1eeapO1XHPjB'
    'K/kK5sCHPkyAi0sn9rMmCUyc+mEc8mwUmF+XTgprY6wot+uNYNLFk0kkyXkCNnIZg2qRS28PWoAc3hIFCWT6WSfdKxNurBeQTexT'
    'VbcQyGtMjraDUXaWi65vAI526SCI1vZaJm3FiVDtBpQADXaMQ9Lzd57/+loQPGYgJ5U18pkbSpsk8HQGHLD63osB4GzipO/PwlaA'
    'DLWvLi5KMzxhmTbGnasDBTwvEneT6y5Yf6tnjrW70naMrOxZmBRgOTllhm7WjxgmUc1hBSF437kPOf6B2CgfNACbZzxPyQtbNSgq'
    'HwizUGdBHs7YQXfbAAUxsX3J04MNYvuscFO+8vTgN4L27A7PEVT0Je0/TpFX2omCT1n3ZgXN06D6WjJnglCvkGcfsHHnT5T5nHea'
    'AON8muXQ09ztgpafxc0ijw47Wi/lEtinH6L62rkRgQ9qNmsXWLX1UVrk2SOWRQydEIh7erbAC1RoEsuC45ORsAfoSO783+z5vxLM'
    'CcPdqUHyuZG07TIVkh1g4MxmuvYLYeCnTvqEomcJeaByDCCQwkzRA9aVkbVPGoGNdhxTnxn5WWGFNbJmMb3jnVxaMXM/9Ijzgyhe'
    'Xp8msA1FYV+KJXxz7ieCYw5rPghriBJ8Y/P7jiT8llkfn4Xl6TGJxH770ymMV3nucy84myoZbyl8E4Jx9tNpl/JtGpthJmQ4Mjiu'
    '2xD5IbHPfw98rYwwgXukJ83g+EjFxXijQ/ftdt3ynMIb4slumNku6wwsE05I+UdJNWMO323xdNjNppwJqeFV1obS9oenqPiU5YaP'
    'O7NTiRoa6vbmD/QgVsRPfykFzMx5mAihZXosbBk45ms1y/OjFklRzNoymobCR8wriMSzfqWZl/haEhBi55n3vKKSqFXmzJsMZjaZ'
    'sykqaumNLMiyn+ed4gC8Q85j7mPA8K9gSuKQ/01psLoX/Xkkj70naIhGRRqCmm2u17zsY56PFg9Pk51sHfFoDF95TZ8P5BM8Ny9i'
    'UZkGfCCs21WC2DxLMaOd3JUGRv4sRjFjWpZjxDhn+k7e3SxEP0zFqEeLOPJfrxcZKbOcaKW+k/d/Qaioi9RNZ0FBgiLuUZHl3Asf'
    'wZMyaaTDmjXtzbHGSlBsGkqKbiJxuNcBzMLHq/I8s5q4nGpaPxrx1E2BpGaH/VQU/gtJsBuUmNCB5XDwZ6LaPmO01ULhyEC4KQwC'
    'HfMx8DP/sJ9wLnmCkE2SKKyRO5Jglxz72ZMGxcZzy2P5M8MMzbTYDdl0fC39OCPrx4+CI9EZBhJMfXKvm9F1P2mRvnLptZNP62VF'
    'glr0DrjeBL/0m2fZuPflU3GjpRf7npFl4vEJnJiQOek5Pz+OadKkmiyEE4TgBCYTTFc2Fm+H8zXfigFrfyGjQnQWlK1JHWHnigcW'
    '3d44mwslcihPL0858mw6hIM0QuIL6/c8rPNcQWYkTO3wiEoGA74XBguWcVGrtxfhNptdG9Z7M1CqHb8RZZJaReyJPE9fesTXzoDs'
    'jhtyYLkN+zTyKsSU2Kx+9b3wUxd+zAqnMz4DVTCRH1FZHHRQ3xJO0ZtlHHFpjmhkonCD3yO09Io/aZD0+TKNCysONPoNAkZerzCH'
    'qaYEaxc/yrn0nzYFZsMySbkv2eoIsTt/SG0BXXjDTFhmuu8yT0/4s8vKwuk+N0f5f3YY08l6myQKrcUyOY6Z6l/TeaZ2oGlZUeeV'
    'aPJXl4X0qCBMcDIviGYy/evBfv7AuhfVLZZlUkJWikOQIv3fE1KoUYSomN7tBmAxL8eECd0m4RDl9jwhKQEM1js5cJaJgagsohbp'
    'vIC58suclPgUuyMJlJ+Ye9bX6EfbSiFHCmaY2nKKrTi3tsi5Goqsn5yDpNdMJU+ztUblMKESpEUYOTKlHoFU0jpZleRyqNjv1k1C'
    'qKOHWY1WM3u6YYgvx7KcU36MQi57sggfEam3h0lLO5/kFGNePyVLhbHsPqwhX8gApXhKOdIKpmgQBu9/28Xzx1rBEhtCdou77bH3'
    '48Dz4oUxSJOn9kEajyVi4O/5pFlSM3t/qPOsVW9KXi2zmtGETGz5vfpE7IRyMpSWAXAoHCjsukXEp1uiaVp2JhdYtej5F2dBfjmk'
    'KlUR+Y5BZVSO8LZ40jE75/2y3yjvdmmXD+Pzr8WSubJz7eoehYkJjVNjY17exqRvLMydqxkXFgIJSWvKMbf2FMxVyar13m4CBmue'
    'BCE1tlSsLew7hqUW3giBJbsM55SBJz6CkUqBGVDEsd/gLnktlkCzfZn7acRHiqeNPWGyDYn5aZ4jYBywxnzsfEIVT/5weWlp6wLC'
    'uINIwODcFAZ7RDA6tmooGMaJJ6wEHXvRrKLcPWkA/+1iIftckKNjxSZLag1ockaJbgrBDxIKh8gcMX1zU2J2hqFERTCHNo7HXarz'
    'kLnnBUPErtStTdcPXAyLphlEPgMQP2kG9oc8pqVHSyFYyJFxECzkbEndEwW48y5PSY48LycAlU9USKdpB+yHnGCRtZAuVdQE9UFh'
    'xAG7KwySCvfjpGIoK7Y6L9UXggUQWVGt9ZVg2uUGWUSeXTJxPesPlm9r+QSin4dD1Kp12kdfPtXc4/UvaY+ilwdGMmIIyJVmIpHC'
    'ahZCaZ2/EkxbSSvnHa2/1Dghqy4a6oP9F2KzfiAP6HZi6Nkia+EBbDlbkzmYz22kUwxsRB9hLWenJP0ys6fT9e6gFJzQPltsAN7Z'
    'TGW7EPkqqbFvBMGgcybGHAt3vmBoH8nSPy4RpEOxmHvS6ZgUo5jOIyMa4lq+zwTxJgp04/wSU58lPCISoYHAjKH4iOksK/vDZf62'
    '+hPHvNdJRr5Gwp8uedcyAmoko7oY2N+TWlgJVqFcNlsCyXLJZVGcFL/vTMSiyhwtTyryVCA/iL+KxKy2GFgi80B4ieU9I4PWwjaJ'
    'YCVzIO+1ZMqfEPM+nkOBPqayodDz3OpH3w1p+GUno3PdGoEJgD8p0OzDnDWL5G0LEDcQCz5KfLolpMMFZ9NlBQSHE7LnPmsDPnuX'
    'EiYZgeiSw4uHy2WxDc6et0ak3eb1JSgmfgxl/b1n4i78uFRRCxJAy/q1WAJshBCUnI1d6ckApEM02CYjRwjZIy53mX9hnOJ3uRdC'
    'imUmueG3rcCe8Mk5hkj6EgwUUi2TqlCOfHWiDtyt8gUh3FdCYPCkUnMlscMeZhG0LGp/vxcDK0r16lhqh5XzZmGWRR4IuymbPFKh'
    'lh4RGK0XRC0uY7OkmPWAaAfMgdbSVTuZj0o9jQBIhTQrW+vsp6yj0ZYDnABxjtjE2MBcgcR2D4X4VVx/4XOU51HdCLA4+TPUiKx2'
    'Iokg/V4jCqVVD+hLtkJIChdWqmTfb5fsAG3+IxALUe7tYCLG/ZVHMG6LgyQHbWtNWbKiRPntq0/oj87L5W6dkVdZT/VeG+DzSMoI'
    'O8viogDud2onpIfgfMT74gDKk6w/UM1CF6mqVWSD7L418PtaK4gd6PKjxFxYN+K3b4IZM3OgLZV8Ucn7YZlatyvh1C3//WSrX6vU'
    'Fk76S/JxUmc7zBRcLocjm0WdmA/MRRRfRJput06mkOQTNusiYRGeL0w+Luepss7hUqgAzGdtkoKVIztQAamAuISTMqrndOhg1vQH'
    'DcA3TxMVcxMbclyiSC9vXpYg53lEMfEVERYBZcUebQ6jaF56IeeFDZhrWiYOggtkHKTQtV5tOWHwzFeW84A/rnZ9yYKS+14U6rqF'
    'MZExovV74QKIidHTzuaSyz4InHYByz1ZbevzhFzh88vcAwLe+540OBkWrbVoZe/bbQJiY09QYcbFj9fGXSq6y4HSsip3TCT/FcJs'
    'tSodt6XBmS93xXcE1dbZinpB+wMlq9uN318yvos8IvMksOhWyHm7AjG7Ud5QyoUf2bVrIfsR+Cw/N64VlIr3ai4ty07iQpuOmWv4'
    '8qGOQG5k3mqRXGipxiFn1ds1ERkUqeZAhH/C/ZKy3FS3myWrWh58zMh7znBLtVbdGNqqNFdWYPW2NECq5KEYqSNVfhswg/0OV6h2'
    '2cnhq1tfVb6Vp2Do3DNjjcocdGYddUWRxwuX9Pi4WYpeq57kJdctbewqkmb9oWENHYmjpvfk0/pR2oGYi9sfMr2aqDArRhneSoLw'
    'gyIHP4p8DBN2i/BfvAxkW1vg8e0WAGLQT9ZKKIu0BbAOHczMazGbzFJXoIryWZbQDCyoh8qfrz5rt4Bn7rVIlpmi0yDGHFzwJbc6'
    'SWKsIGQAEHvnGCRaftIiGUGKLct8vCoK7c+1WKq8vsXUIAcuaqg2gm1czKhDLuY7ObCJ+LgPRyvj2nkpoJpuw9ZRo27WV3gVOMPY'
    'F+/3i75dHEtNA0WV3LbMKnTAL2YF3MleVtOuCB7S1V1n4Bgg0bwwtRAMAc+la1lowSw2mOD6YTsw5NMhUT4jmLBktK42Dur3aEMJ'
    'q3kcl1rl/KdbuFfkxdhIms2t/CmL57+TBsaJWnqJKiqldHKXz5JCrOCCzrVu1Kr95PBuUM9jCLH5JyxSsfm7EVqlUZSgPZnz26Wt'
    'pkv3Kdwj1+uuZH9Uav+AGxOvb1Ih+t+SBCWQlNCrMc+SvGyyT+LTQTt7a3284qov7/IJJpGtShOujIk3ddSNPCX5gd8JoIXfEtXe'
    'adc9U3ISx1SJIdp7q2wmfrgfJxhsr4NaSnFUDFuvlXWFVzlaFefjHm09dSxMkEoAJdDjsNtvo/ce3G11qPhR3JIITevN1Zh7e0HK'
    'QRjOs/cBoGQb2fiA2ZorrEPwViy5yQeZ+IIIDdkljk/PUv/29yY3x/Eg4lRDXr0wlhcDVQz9jmtdpnUQo5ILz91T19cvt+VT4Yss'
    'evJKDn73uaDoUH4jnove26SfrYPuzy1JANQdNE8Btpo8oigutLkW18K8kUpnx8HEwsZaTndUwv5U9FXHVlR2rJLYfbtdgtiOYUBu'
    'NiAIWIPZZQ0CEmTCGsoXuNfGjnF4+jxyVUUQ11hbwZi9FfCwEEjm4sF7Xmj9Eh4n4LZrm/neWNd83s+5Bdw/kRlYqd97VzoZnQcX'
    '/NyL+gFufi/roD5CznKWN/hOLuErx6iPRvBVX7AeIyAt1SB/PDyjONRQqUiRsuhILDHuGeyNniam0PVmldrHzp4Jtdxpu6vphBWs'
    'Yzq27sONTiw/m/j+aE1/0ABs6Wk/KcfuwlUG9T07fgU3Izc+Y+5nZAk598OmA4REvmfcXa341jEzvqiSKQygnYyUS9wRTDL0636b'
    'lEN/zEo/vlL9HtuDIaL4bdRW3cQ8FnPvGCR6JdMKlWNwLmzDvjp2Wtyh9/BmVbxrTltH/hLUJsB87BOdemzEEHA7qH6Zn5Qn5YrF'
    'sLbxcZgrNqYqKBwA/HTLc8r7fC8GTzQhBWS1xKIz4PJNY9NdElzu11qZkhaHV1id9tgOpUVC3d9K2fab8uDwNwGh/E7lSBdMQYFv'
    'cIPoV+P1J85FBnkhq9j5tw+zeZ/RoMxy0D9sBCJyhxRnVjIamW/2Y37R1AWMZdWD/+8bgYFU2sN6cYNqyM2087JmVRfvCsPH7Tup'
    'YPnGf3isUQV01411p6RU5VXUMHQZU0x9jktrA2s3RGZX0tYuHE0K/2rzQI7zpOdeZ7KWV9R7TRXSpF6dAGcvjqOv31AbbtNGFbW/'
    'bXW2xmuepBd7XuEuqVAzjt+WquKupn+0b849yY63ogdgYSGveCNKoCA2uqJUoOSMUBvWlYJ12So+aHgVnOObwoDAa56H717MHD1X'
    '3STCQJ7x3fD1TjqoqR6ngu7nmgp1ZWPxSQMQ2TupWvJUcBcH2Fw8KTmQX82R5/yAmBOwrv2ejjrksvcsvNudJ5cDldV7uEAuE3Ve'
    'eLuU+Tn1fUkscLe3phRRPz4mMNPqRrs2Z3ldqETKIib14ypBd0RTKOGc6TSwdn6Mt/MgORCySe5Kw/mf+XwcWQE3BFeY+5xcfDVa'
    'rN+4fNstcRBMOOdxROm/l61f84iM7iz/0lcazYOynzcDoZRzFie6h3BrFkd3l7MXVULeiCY38pwVW3B6AigZ7WIJZmN6Un5+VxFe'
    'i/Nxsr4GRIZyOb7Of6X0eFUEvKagQsriIANXpMVcnCvmQr6oIHWKcVnGBnXCOdAxhBMC8L6YFIbr/z8NwZYSJ+1zLzhkQDueZexp'
    'WzMv+LUcGDs1cD5WeOgXInQ8U1p3M6J+qlH7J/mlSTF/M0+IubI7k14LAszkPMZj38v8E5sSEnyNVIIJ1TbBL3ws47AUF5+53J1H'
    'vMHrStkqX23Ff+wjp9UC1TIWA3f1TYKR8l1Ls7zVANjR5yFQFXVwK9nvkFAcL3D3lESQ5PJZk4TSnCejQlRr+qWL3GV16O6Jgq1w'
    'DofiefaapVzGfEEXBqGYeo/n6ORKLNXqwa6+QEYoCqWmcAmZ2y3sLPmz6pym/M5kN82JhwU5jF8tBsx5LnRUlciQK24Uq/RYrVl1'
    '3f/1fwIdg0E='
)
VIS400_REF = {}
try:
    _raw = zlib.decompress(base64.b64decode(REF_B64))
    if hashlib.sha256(_raw).hexdigest() == REF_SHA256:
        VIS400_REF = json.loads(_raw.decode())
        log('vis400 reference table: %d molecules | sha256 OK | hit1 %d | in25 %d' % (
            len(VIS400_REF), sum(1 for v in VIS400_REF.values() if v.get('rk') == 1),
            sum(1 for v in VIS400_REF.values() if v.get('rk', 0) > 0)))
    else:
        log('[warn] vis400 reference payload sha mismatch -> metrics disabled')
except Exception:
    traceback.print_exc()
    VIS400_REF = {}


def _matched(smi, ref):
    """exact SMILES or an equivalent InChI (stereo-insensitive) match against the known answers"""
    if not smi:
        return False
    if smi == ref['ans'] or smi == ref['gt']:
        return True
    if not RDKIT_OK:
        return False
    try:
        return _ik14(smi) in _ik14(ref['ans'])
    except Exception:
        return False


_IK_CACHE = {}


def _ik14(smi):
    if smi in _IK_CACHE:
        return _IK_CACHE[smi]
    k = None
    mo = Chem.MolFromSmiles(smi) if smi else None
    if mo is not None:
        try:
            k = Chem.MolToInchiKey(mo)[:14]
        except Exception:
            k = None
    _IK_CACHE[smi] = k
    return k


def metrics(lists, label, ids=None):
    """vis400 MRR@25 over the ids we can identify (the 400 visible molecules), straight from the
    live lists. Returns None on the hidden test set (ids do not overlap -> nothing to score)."""
    ids = [m for m in (ids or sorted(lists)) if m in VIS400_REF]
    if not ids:
        return None
    rr, t1, t25 = 0.0, 0, 0
    for mid in ids:
        cands = [s for s in (lists.get(mid) or []) if s][:CFG['TOP_K_SUBMIT']]
        ref = VIS400_REF[mid]
        for rank, smi in enumerate(cands, 1):
            if _matched(smi, ref):
                rr += 1.0 / rank
                t25 += 1
                if rank == 1:
                    t1 += 1
                break
    out = dict(label=label, n=len(ids), mrr25=round(rr / len(ids), 5), top1=t1, truth_in25=t25,
               hit1_truths=int(sum(1 for m in ids if VIS400_REF[m].get('rk') == 1)),
               in25_truths=int(sum(1 for m in ids if VIS400_REF[m].get('rk', 0) > 0)))
    MENU.append(out)
    log('MENU | %-42s vis400 MRR@25 = %.5f | top1 %3d | truth_in25 %3d | n %d' % (
        out['label'], out['mrr25'], out['top1'], out['truth_in25'], out['n']))
    return out


def recall_audit(label, ids=None):
    """how many of the 40 gen-p2 rank-1 truths / 209 in-25 truths are INSIDE cands_by_mid"""
    ids = [m for m in (ids or sorted(mols)) if m in VIS400_REF]
    if not ids:
        return None

    def _hit(cands, ids_):
        return sum(1 for m in ids_ if any(_matched(s, VIS400_REF[m]) for s in (cands.get(m) or [])))

    hit1 = [m for m in ids if VIS400_REF[m].get('rk') == 1]
    in25 = [m for m in ids if VIS400_REF[m].get('rk', 0) > 0]
    # flatten cands_by_mid to SMILES lists for the audit
    cand_smis = {mid: [s for s in cands_by_mid.get(mid, [])] for mid in ids}
    out = dict(
        label=label,
        native_pool_hit1='%d/%d' % (_hit(cand_smis, hit1), len(hit1)),
        native_pool_in25='%d/%d' % (_hit(cand_smis, in25), len(in25)),
        n_cands=int(sum(len(v) for v in cands_by_mid.values())),
        n_mols=int(len(cands_by_mid)),
    )
    log('RECALL | %r' % (out,))
    return out


RECALL_AUDIT = recall_audit('native pool (zero pruning)', list(mols))


In [ ]:
# ===== STAGE B: GLACIER on the native pool (2 workers × 2 threads) =====
gl_scores = {}
GL_STATS = dict(status='skipped')
with stage('Stage B: GLACIER native pool (%d items, budget %.0fs)' % (N_ITEMS, CFG['GL_BUDGET_S']),
           CFG['GL_BUDGET_S'] + 300.0):
    if GL_PKG and ICE_PKG and items:
        try:
            sys.path.insert(0, GL_PKG)
            from gl_fuse import run_gl, rerank_multi as _rerank_multi
            rerank_multi = _rerank_multi
            t_gl = time.time()
            gl_scores = _run_gl_parallel(items, GL_PKG, ICE_PKG, WORK,
                                         os.path.join(WORK, 'ice_site'),
                                         CFG['GL_BUDGET_S'], N_WORKERS)
            GL_STATS = dict(seconds=round(time.time() - t_gl, 1))
        except Exception:
            traceback.print_exc()
            gl_scores = {}
    else:
        log('[warn] GLACIER skipped (pkg=%s/%s items=%d)' % (bool(GL_PKG), bool(ICE_PKG), len(items)))

GL_N_SCORED, GL_N_MOLS_SCORED = _count_real_scores(gl_scores)
log('Stage B GLACIER final: covered %d | scored molecules %d | finite scores %d' % (
    len(gl_scores), GL_N_MOLS_SCORED, GL_N_SCORED))


# GLACIER-only ranking for diagnostics
def gl_only_order(mid):
    base = base_rank(mols[mid])
    smis = [c['smiles'] for c in base]
    forms = [c['formula'] for c in base]
    scs = [float(c['score']) for c in base]
    order = list(range(len(smis)))
    gl_m = gl_scores.get(mid, {})
    try:
        if rerank_multi is not None and gl_m:
            order = rerank_multi(smis, smis, scs, forms, [{}, gl_m],
                                 [CFG['LAM_ICE'], CFG['LAM_GL']], top_n=CFG['TOP_N_RESCORE'])
    except Exception:
        traceback.print_exc()
    return [smis[i] for i in order]


metrics({mid: gl_only_order(mid)[:CFG['TOP_K_SUBMIT']] for mid in mols},
        'stage3a_B GLACIER only (native pool)', list(mols))
write_sub([(mid, order_to_25(gl_only_order(mid), list(range(len(gl_only_order(mid))))))
           for mid in sorted(mols)], 'stage3a_gl_native', len(mols))
log('STAGE B done: stage3a_gl_native written')


In [ ]:
# ===== STAGE C: ICEBERG on the SAME native pool (2 workers × 2 threads) =====
ice_scores = {}
ICE_STATS = dict(status='skipped')
with stage('Stage C: ICEBERG native pool (%d items, budget %.0fs)' % (N_ITEMS, CFG['ICE_BUDGET_S']),
           CFG['ICE_BUDGET_S'] + 300.0):
    if ICE_PKG and items:
        try:
            sys.path.insert(0, ICE_PKG)
            from fuse import run_ice, rerank as _rerank
            rerank = _rerank
            t_ice = time.time()
            ice_scores = _run_ice_parallel(items, ICE_PKG, GL_PKG, WORK,
                                           os.path.join(WORK, 'ice_site'),
                                           CFG['ICE_BUDGET_S'], N_WORKERS)
            ICE_STATS = dict(seconds=round(time.time() - t_ice, 1))
        except Exception:
            traceback.print_exc()
            ice_scores = {}
    else:
        log('[warn] ICEBERG skipped (pkg=%s items=%d)' % (bool(ICE_PKG), len(items)))

ICE_N_SCORED, ICE_N_MOLS_SCORED = _count_real_scores(ice_scores)
log('Stage C ICEBERG final: covered %d | scored molecules %d | finite scores %d' % (
    len(ice_scores), ICE_N_MOLS_SCORED, ICE_N_SCORED))


In [ ]:
# ===== STAGE D: M-S12 rerank_multi on FULL base_rank -> M-S13 A1 =====
G2_RANKED = {}
for mid in sorted(mols):
    m = mols[mid]
    base = base_rank(m)
    smis = [c['smiles'] for c in base]
    forms = [c['formula'] for c in base]
    scs = [float(c['score']) for c in base]
    order = list(range(len(smis)))
    ice_m = ice_scores.get(mid, {})
    gl_m = gl_scores.get(mid, {})
    try:
        if rerank_multi is not None and (ice_m or gl_m):
            order = rerank_multi(smis, smis, scs, forms, [ice_m, gl_m],
                                 [CFG['LAM_ICE'], CFG['LAM_GL']], top_n=CFG['TOP_N_RESCORE'])
        elif rerank is not None and ice_m:
            order = rerank(smis, smis, scs, forms, ice_m, lam=CFG['LAM_ICE'],
                           top_n=CFG['TOP_N_RESCORE'])
    except Exception:
        traceback.print_exc()
        order = list(range(len(smis)))
    G2_RANKED[mid] = order_to_25(smis, order)

metrics(G2_RANKED, 'stage3b_genp2_native (M-S12 full base_rank)', list(mols))
write_sub([(mid, G2_RANKED[mid]) for mid in sorted(mols)], 'stage3b_genp2_native', len(mols))
log('STAGE D part 1 done: stage3b_genp2_native written')

A1_DIAG = dict(molecules=0, rank1_frozen=0, body_resorted=0, n_sig_calls=0)
A1_LISTS = {}
for mid in sorted(mols):
    g2 = G2_RANKED[mid]
    if not g2:
        A1_LISTS[mid] = order_to_25(FALLBACK, list(range(len(FALLBACK))))
        continue
    head, body = g2[0], list(g2[1:])
    try:
        body = sorted(body, key=lambda s: -sig_ms2(mid, s))
    except Exception:
        traceback.print_exc()
    a1 = ([head] + body)[:CFG['TOP_K_SUBMIT']]
    A1_LISTS[mid] = order_to_25(a1, list(range(len(a1))))
    A1_DIAG['molecules'] += 1
    A1_DIAG['rank1_frozen'] += int(bool(head) and head == g2[0])
    A1_DIAG['body_resorted'] += int(body != g2[1:])

A1_DIAG['n_sig_calls'] = int(sum(len(v) for v in SIG_CACHE.values()))
A1_DIAG['rank1_identical_to_genp2'] = '%d/%d' % (A1_DIAG['rank1_frozen'], A1_DIAG['molecules'])
A1_DIAG['set_identical_to_genp2'] = '%d/%d' % (
    sum(1 for mid in A1_LISTS if set(A1_LISTS[mid]) == set(G2_RANKED.get(mid, []))), len(A1_LISTS))
log('M-S13 A1: %r' % (A1_DIAG,))
metrics(A1_LISTS, 'stage4_A1 final (M-S13, m=0)', list(mols))
write_sub([(mid, A1_LISTS[mid]) for mid in sorted(mols)], 'stage4_A1_final', len(mols))
log('STAGE D part 2 done: stage4_A1_final written')


In [ ]:
# ===== report + submission contract =====
try:
    _short = [mid for mid, lst in A1_LISTS.items() if len(lst) < CFG['TOP_K_SUBMIT']]
    if _short:
        _extra = []
        for _i in np.argsort(-P_NPL):
            _s = P_SMI[_i]
            if _s not in _extra:
                _extra.append(_s)
            if len(_extra) >= CFG['TOP_K_SUBMIT'] * 8:
                break
        _extra += [s for s in FLOOR_SMILES if s not in _extra]
        _fixed = {}
        for mid, lst in A1_LISTS.items():
            if len(lst) < CFG['TOP_K_SUBMIT']:
                lst = (list(lst) + [s for s in _extra if s not in lst]
                       + list(FALLBACK))[:CFG['TOP_K_SUBMIT']]
            _fixed[mid] = lst
        A1_LISTS = _fixed
        write_sub([(mid, A1_LISTS[mid]) for mid in sorted(A1_LISTS)], 'stage4b_topup', len(A1_LISTS))
        log('top-up: %d molecules needed extra candidates' % len(_short))
except Exception:
    traceback.print_exc()

CHECK = dict(path=SUB)
try:
    back = pd.read_csv(SUB, dtype=str).fillna('')
    k = back['smiles'].str.split(';').map(lambda v: len([x for x in v if x]))
    bad = []
    if RDKIT_OK:
        for row in back['smiles']:
            for s in str(row).split(';'):
                if s and Chem.MolFromSmiles(s) is None:
                    bad.append(s)
                    if len(bad) >= 5:
                        break
    CHECK.update(rows=int(len(back)), header=list(back.columns),
                 unique_ids=int(back['molecule_id'].nunique()),
                 cands_min=int(k.min()) if len(k) else 0,
                 cands_max=int(k.max()) if len(k) else 0,
                 cands_mean=round(float(k.mean()), 2) if len(k) else 0.0,
                 empty_cells=int((back == '').sum().sum()),
                 unparseable=bad[:5],
                 bytes=int(os.path.getsize(SUB)) if os.path.exists(SUB) else 0)
    CHECK['contract_ok'] = bool(
        CHECK['header'] == ['molecule_id', 'smiles'] and CHECK['rows'] > 0
        and CHECK['unique_ids'] == CHECK['rows'] and CHECK['cands_min'] == 25
        and CHECK['cands_max'] == 25 and CHECK['empty_cells'] == 0
        and not CHECK['unparseable'])
except Exception:
    traceback.print_exc()
    CHECK['contract_ok'] = False

COVERAGE = dict(
    gl_covered=int(len(gl_scores)), gl_mols_scored=int(GL_N_MOLS_SCORED),
    gl_n_scored=int(GL_N_SCORED),
    ice_covered=int(len(ice_scores)), ice_mols_scored=int(ICE_N_MOLS_SCORED),
    ice_n_scored=int(ICE_N_SCORED),
    native_pool_n_cands=int(N_CANDS_NATIVE),
    native_pool_n_mols=int(N_MOLS_NATIVE),
    full_gl_cov=bool(GL_N_MOLS_SCORED >= N_MOLS_NATIVE * 0.99),
    full_ice_cov=bool(ICE_N_MOLS_SCORED >= N_MOLS_NATIVE * 0.99),
)
log('COVERAGE: %r' % (COVERAGE,))

RPT = dict(
    kernel='dcvxzyk/enveda-r45-unpruned-mp',
    round=45,
    arm='unpruned native pool (gen-p2 v8) + 2-worker CPU multi-process GLACIER/ICEBERG + A1 (m=0)',
    gpu=bool(HAS_GPU), torch=TORCH_VER,
    config={k: v for k, v in CFG.items()},
    n_workers=int(N_WORKERS), omp_per_worker=int(OMP_PER_WORKER),
    n_molecules=int(len(mols)), n_spectra=int(sum(len(v['spectra']) for v in mols.values())),
    median_formulas=float(np.median([len(m['formulas']) for m in mols.values()])) if mols else None,
    median_cands=float(np.median([len(m['cands']) for m in mols.values()])) if mols else None,
    n_gen=int(sum(sum(1 for c in m['cands'] if c['src'] == 'gen') for m in mols.values())),
    n_cands_native=int(N_CANDS_NATIVE),
    n_mols_native=int(N_MOLS_NATIVE),
    n_items=int(N_ITEMS),
    coverage=COVERAGE,
    recall_audit=RECALL_AUDIT,
    gl_stats=GL_STATS, ice_stats=ICE_STATS,
    menu=MENU, a1=A1_DIAG, contract=CHECK, stages=STAGES, writes=WRITES,
    write_stages=['stage1_floor', 'stage2_base', 'stage3a_gl_native',
                  'stage3b_genp2_native', 'stage4_A1_final'],
    vis400_reference=dict(genp2_v8=0.16988, a1=0.18955, top1=40, truth_in25=209),
    submission_bytes=int(os.path.getsize(SUB)) if os.path.exists(SUB) else 0,
    runtime_seconds=round(time.time() - T0, 1),
)
try:
    with open(os.path.join(WORK, 'r45_report.json'), 'w') as f:
        json.dump(RPT, f, indent=1, default=str)
except Exception:
    traceback.print_exc()

FINAL = [m for m in MENU if m['label'].startswith('stage4')]
log('R45 DONE in %.1fs | submission.csv %s bytes | contract_ok=%s | GL mols_scored=%d | ICE mols_scored=%d | vis400 %s' % (
    RPT['runtime_seconds'], RPT['submission_bytes'], CHECK.get('contract_ok'),
    GL_N_MOLS_SCORED, ICE_N_MOLS_SCORED, FINAL[-1]['mrr25'] if FINAL else None))
log('STAGE MENU')
for _row in MENU:
    log('  %-42s MRR@25 %.5f | top1 %3d | in25 %3d' % (
        _row['label'], _row['mrr25'], _row['top1'], _row['truth_in25']))
log('WRITE LEDGER: %r' % (WRITES,))
log('STAGE LEDGER: %r' % (STAGES,))
